# 上下文工程

提示词工程是一个子集。上下文工程才是目标。提示词是你输入的字符串。上下文是所有进入到模型窗口的东西：系统指令、检索文档、工具定义、会话历史、少样本例子，然后才是提示词。2026最好的AI工程师就是上下文工程师，他们决定哪些东西进入，哪些东西屏蔽，以什么顺序。

## 问题描述

前沿模型现在都宣称1M窗口大小，但实际上系统提示词、检索到的文档、工具定义就能占去不少，再加上多轮的会话历史，并没有想象中那么富裕。

更重要的是，注意力对上下文长度的开销增长很快，原生的注意力机制是指数级别，现在会使用更高效的变体。

除此以外，“大海捞针”测试表明LLM在检索信息的时候，对开头和结尾处的信息检索效果好，对中间信息不尽人意。

总的来说，上下文工程就是维护窗口中信息与噪声的比率。

## 基本概念

### 上下文窗口是稀缺资源

系统提示词、工具定义、检索文档、会话历史、少样本例子、用户输入、输出预留。上下文窗口需要平衡上述的内容来达到最好的任务性能。

### 中间衰减

关于上下文工程最重要的经验发现。模型更容易关注到在处理开头和结尾处的信息，但是对于中间的内容注意力分数会偏低，导致被忽视。

这有一些工程启发：
- 把最重要的信息放在开头（系统提示词，关键指令）
- 把最近的查询和最相关的上下文放在最后（利用邻近偏差）
- 将上下文的中间部分当作低优先级区
- 如果你必须在中间包含信息，在结尾处重申关键点。

### 上下文组件

#### 系统提示词

人格、约束以及行为准则定义。

#### 工具定义

要有选择性的进行召回，不要一股脑塞进去

#### 检索上下文

宁缺毋滥，不相关的检索结果只会引入噪声，误导模型

#### 会话历史

每轮次的用户输入和模型回答。线形增长且大部分与当前轮次的会话无关。

#### 少样本例子

2-3个高质量的例子用来演示期望行为比成千上万词元的指令更有效

#### 生成预留

上下文需要为模型生成预留一定的空间，通常2000-4000

### 上下文压缩策略

#### 历史摘要

当历史超过某个阈值时运行摘要。

#### 关联性过滤

当你检索到的10篇文档中只有3篇关联度比较高的话，大胆放弃剩余的7篇。

#### 工具裁剪

不要添加不必要的工具，要识别用户的查询意图然后只引入与意图相关的工具

#### 递归摘要

对于长文档，先整体摘要、再章节摘要...

### 记忆系统

包含三种时间跨度

#### 短期记忆

只在当前会话中，存储在上下文里面。通过摘要或者剪枝管理。

#### 长期记忆

各个会话中都需要遵循的一些偏好，存储在数据库中，会话开始时召回。比如Claude Code 的 CLAUDE.md

#### 情景记忆

可能相关的特定过往交互。靠嵌入向量存储，然后靠相似度做召回。

### 组装动态上下文

直觉是：不同的查询需要不同的上下文。使用固定的系统提示词+固定工具+固定历史很浪费。考虑每次查询时动态组装.
- 对查询意图分类
- 选择相关工具而不是所有
- 召回相关文档而不是固定的几何
- 保留相关的历史轮次而不是所有
- 针对任务类型添加少样本例子
- 将所有的内容按顺序组织：最关键的在最前，重要的在最后，其他的在中间。

# 开始编码

In [ ]:
def reorder_lost_in_the_middle(items, scores):
    paired = sorted(zip(scores, items), reverse=True)
    sorted_items = [
        item for _, item in paired
    ]
    if len(sorted_items) <= 2:
        return sorted_items

    first_half = sorted_items[::2]
    second_half = sorted_items[1::2]

    return first_half + second_half